# CP3 - Transformaciones I

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Conocer SparkSQL y los DataFrames.
- Aprender a leer y escribir datos.
- Comprender los tipos de datos en PySpark.
- Trabajar con columnas en DataFrames.
- Aplicar transformaciones sobre los datos.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

sc = spark.sparkContext

DATA_PATH = "/home/jovyan/work/data_practicas_1/"

## 1 - Spark SQL

Se trata de una API estructurada de Spark. Para realizar el procesamiento de datos es necesario un formato y una lógica de cálculo, estructurados.

Spark SQL está compuesto por:
- DataFrame API.
- Catalyst Optimizer.

## 2 - Dataframes en PySpark

### 2.1 Crear un Dataframe

**Crear un Dataframe a partir de un RDD**

Para ello, primero debemos crear un RDD. Una vez hecho, usamos la función `toDF(columnas)`, pasándole como argumento una lista con el nombre de las columnas.

In [2]:
rdd = sc.parallelize([(i, i**2) for i in range(10)])
rdd.collect()

[(0, 0),
 (1, 1),
 (2, 4),
 (3, 9),
 (4, 16),
 (5, 25),
 (6, 36),
 (7, 49),
 (8, 64),
 (9, 81)]

In [3]:
df = rdd.toDF(['numero', 'cuadrado'])

Para conocer el esquema del Dataframe (los DF en Spark tienen un schema asociado), podemos usar la función `printSchema()`. En esta función vemos lo siguiente:

- Nombre de la columna: ej. "numero".
- Tipo de dato: ej. "long".
- Nullable: true o false.

In [4]:
df.printSchema()

root
 |-- numero: long (nullable = true)
 |-- cuadrado: long (nullable = true)



Con la función `.show()` podemos visualizar los primeros `n` elementos del dataframe. Por defecto serán 20.

In [5]:
df.show(5,0)

+------+--------+
|numero|cuadrado|
+------+--------+
|0     |0       |
|1     |1       |
|2     |4       |
|3     |9       |
|4     |16      |
+------+--------+
only showing top 5 rows



### 2.2 Crear un Schema

In [6]:
rdd1 = sc.parallelize([(1,'Jose',9.5),(2,'Teresa',6.7),(3,'Katia',8.8)])

Lo primero que vamos a hacer, es ***crear nuestro propio schema***, usando dos formas diferentes:

In [7]:
#from pyspark.sql.types import StructType, StructField
#from pyspark.sql.types import IntegerType, StringType, DoubleType

*Forma 1* - Usando `StructType()` y `StructField()`.

La función `StructField` debe incluir:
- Nombre de la columna.
- Tipo de dato.
- Nullable o no.

In [8]:
esquema1 = T.StructType(
    [
      T.StructField('id', T.IntegerType(), True),
      T.StructField('nombre', T.StringType(), True),
      T.StructField('nota', T.DoubleType(), True)
    ]
)

*Forma 2* - Usando un string de la siguiente forma:

- Caracteres especiales `` para el nombre de la columna.
- El tipo de dato en mayúscula.

In [9]:
esquema2 = "`id` INT, `nombre` STR, `nota` DOUBLE"

Ahora ya podemos pasarlo a DF, indicando como parámetro el schema que hemos creado. Los esquemas `esquema1` y `esquema2` son equivalentes y por tanto, da igual usar uno u otro.

In [10]:
df1 = spark.createDataFrame(rdd1, schema=esquema1)

In [11]:
df1.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- nota: double (nullable = true)



In [12]:
df1.show()

+---+------+----+
| id|nombre|nota|
+---+------+----+
|  1|  Jose| 9.5|
|  2|Teresa| 6.7|
|  3| Katia| 8.8|
+---+------+----+



### 2.3 Dataframe desde una Fuentes de Datos

#### 5.3.1 Leer Datos

La clase **DataFrameReader** lleva a cabo las funciones de lectura de datos. La instancia de la clase se crea de la siguiente forma: `spark.read`

Patrón común de interacción:

`spark.read.format("extension").option('key','value').schema(esquema).load("ruta")`

Versión simple

`spark.read.<formato>('path')`

In [13]:
df_csv = spark.read.csv(DATA_PATH + 'dataCSV.csv')
df_csv.show(5, truncate=False)

+-----------+-------------+--------------------------------------------------------------+---------------------+-----------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
df_csv1 = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'dataCSV.csv'))

In [15]:
df_csv1.show(5, truncate=False)

+-----------+-------------+--------------------------------------------------------------+---------------------+-----------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#### 5.3.2 Escribir Datos

La clase **DataFrameWriter** lleva a cabo las funciones de escritura de datos. La instancia de la clase se crea de la siguiente forma: `df.write`

Patrón común de interacción:

`df.write.format("extensión").mode(...).option(...).partitionBy(...).bucketBy(...).sortBy(...).save("ruta")`

Versión simple

`df.write.<formato>('path)`

Los modos de escritura son: `append`, `overwrite`, `error`, `errorIfExist`, `default`, `ignore`.

In [16]:
df_save = spark.createDataFrame([(1,'Jose',9.5),(2,'Teresa',6.7),
                                 (3,'Katia',8.8)],['id','nom','nota'])

In [17]:
df_save.show()

+---+------+----+
| id|   nom|nota|
+---+------+----+
|  1|  Jose| 9.5|
|  2|Teresa| 6.7|
|  3| Katia| 8.8|
+---+------+----+



In [18]:
df_save.write.csv(DATA_PATH + 'csv2.csv')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/jovyan/work/data_practicas_1/csv2.csv already exists. Set mode as "overwrite" to overwrite the existing path.

### 2.4 Trabajar con Columnas

In [19]:
df3 = df_csv1
df3.show(5, truncate=False)

+-----------+-------------+--------------------------------------------------------------+---------------------+-----------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
df3.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



**Nombre y Tipo de Dato de una Columna**

Usamos la estructura: `df.select('<nombre columna>')`

In [21]:
df3.select('title')

DataFrame[title: string]

**Seleccionar una o varias columnas**

In [22]:
# Forma 1
df3.select('title').show(5) # 1 columna
df3.select('title', 'video_id').show(5) # + de 1 columna

+--------------------+
|               title|
+--------------------+
|WE WANT TO TALK A...|
|The Trump Preside...|
|Racist Superman |...|
|Nickelback Lyrics...|
|I Dare You: GOING...|
+--------------------+
only showing top 5 rows

+--------------------+-----------+
|               title|   video_id|
+--------------------+-----------+
|WE WANT TO TALK A...|2kyS6SvSYSE|
|The Trump Preside...|1ZAPwfrtAFY|
|Racist Superman |...|5qpjK5DgCt4|
|Nickelback Lyrics...|puqaWrEC7tY|
|I Dare You: GOING...|d380meD0W0M|
+--------------------+-----------+
only showing top 5 rows



In [23]:
# Forma 2
df3.select(F.col('title')).show(5) # 1 columna
df3.select(F.col('title'), F.col('video_id')).show(5) # + de 1 columna

+--------------------+
|               title|
+--------------------+
|WE WANT TO TALK A...|
|The Trump Preside...|
|Racist Superman |...|
|Nickelback Lyrics...|
|I Dare You: GOING...|
+--------------------+
only showing top 5 rows

+--------------------+-----------+
|               title|   video_id|
+--------------------+-----------+
|WE WANT TO TALK A...|2kyS6SvSYSE|
|The Trump Preside...|1ZAPwfrtAFY|
|Racist Superman |...|5qpjK5DgCt4|
|Nickelback Lyrics...|puqaWrEC7tY|
|I Dare You: GOING...|d380meD0W0M|
+--------------------+-----------+
only showing top 5 rows



### 2.5 Cambiar tipos de datos (o nombre) de una o varias columnas

**Para una columna**

In [24]:
df4 = df3
df4 = df3.withColumn('likes', df3['likes'].cast('long'))
df4.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: long (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



**Para varias columnas**

In [25]:
df4 = df3
df4 = (df3.withColumn('likes', df3['likes'].cast('long'))
      .withColumn('dislikes', df3['dislikes'].cast('long'))
      .withColumn('comment_count', df3['comment_count'].cast('long')))
df4.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: long (nullable = true)
 |-- dislikes: long (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



### 2.6 Transformaciones en DataFrames

##### **Función `select()`**

In [26]:
df3.select(F.col('video_id')).show(5)

+-----------+
|   video_id|
+-----------+
|2kyS6SvSYSE|
|1ZAPwfrtAFY|
|5qpjK5DgCt4|
|puqaWrEC7tY|
|d380meD0W0M|
+-----------+
only showing top 5 rows



Si queremos crear expresiones a la hora de usar columnas, se puede usar la función `col()`. Por ejemplo, podemos crear una nueva columna calculada y además renombrarla usando la función `alias()`.

In [27]:
df3.select(
    F.col('likes'), F.col('dislikes'),
    (F.col('likes') - F.col('dislikes')).alias('aceptacion')
).show(5)

+------+--------+----------+
| likes|dislikes|aceptacion|
+------+--------+----------+
| 57527|    2966|   54561.0|
| 97185|    6146|   91039.0|
|146033|    5339|  140694.0|
| 10172|     666|    9506.0|
|132235|    1989|  130246.0|
+------+--------+----------+
only showing top 5 rows



##### **Función `selectExpr()`**

Nos permite añadir expresiones dentro de la selección y asignarle un alias. Todo ello usando cadenas de texto.

In [28]:
df3.selectExpr(
    'likes',
    'dislikes',
    '(likes - dislikes) as aceptacion'
).show(5)

+------+--------+----------+
| likes|dislikes|aceptacion|
+------+--------+----------+
| 57527|    2966|   54561.0|
| 97185|    6146|   91039.0|
|146033|    5339|  140694.0|
| 10172|     666|    9506.0|
|132235|    1989|  130246.0|
+------+--------+----------+
only showing top 5 rows



In [29]:
df3.selectExpr('count(distinct(video_id)) as videos').show()

+------+
|videos|
+------+
|  6837|
+------+



##### **Función `filter()`**

In [30]:
(df3.filter(F.col('video_id') == '2kyS6SvSYSE')
  .select(
      F.col('video_id'), F.col('title'), F.col('likes')
  ).show(5, truncate=False))

+-----------+----------------------------------+-----+
|video_id   |title                             |likes|
+-----------+----------------------------------+-----+
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|57527|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|88099|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|91111|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|92831|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|94303|
+-----------+----------------------------------+-----+
only showing top 5 rows



In [31]:
(df4.filter((F.col('video_id') == '2kyS6SvSYSE') & (F.col('likes') > 90000))
  .select(
      F.col('video_id'), F.col('title'), F.col('likes')
  ).show(5, truncate=False))

+-----------+----------------------------------+-----+
|video_id   |title                             |likes|
+-----------+----------------------------------+-----+
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|91111|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|92831|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|94303|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|95587|
|2kyS6SvSYSE|WE WANT TO TALK ABOUT OUR MARRIAGE|96321|
+-----------+----------------------------------+-----+



##### **Función `where()`**

In [32]:
df3.select(
    F.col('title'), F.col('likes')
).where(F.col('likes') >= '50000').show(5)

+--------------------+-----+
|               title|likes|
+--------------------+-----+
|WE WANT TO TALK A...|57527|
|The Trump Preside...|97185|
|2 Weeks with iPho...| 9763|
|Dion Lewis' 103-Y...|  655|
|Which Countries A...| 7848|
+--------------------+-----+
only showing top 5 rows



Vemos que no ha podido realizar el `where()` correctamente porque el tipo de dato es un string. Cambiamos a tipo de dato `long`.

In [33]:
df3 = df3.withColumn('likes', df3['likes'].cast('long'))
df3.select(
    F.col('title'),
    F.col('likes')
).where(F.col('likes') >= 50000).show(5)

+--------------------+------+
|               title| likes|
+--------------------+------+
|WE WANT TO TALK A...| 57527|
|The Trump Preside...| 97185|
|Racist Superman |...|146033|
|I Dare You: GOING...|132235|
|Marshmello - Bloc...|114188|
+--------------------+------+
only showing top 5 rows



##### **Función `distinct()`**

In [34]:
df5 = df3.distinct()

print(f'Conteo del df3: {df3.count()}')
print(f'Conteo del df5: {df5.count()}')
print(f'Se han eliminado {df3.count()-df5.count()} filas.')

Conteo del df3: 48137
Conteo del df5: 41497
Se han eliminado 6640 filas.


##### **Función `dropDuplicates()`**

Elimina aquellos registros duplicados en base a una o varias columnas que seleccionemos. A diferencia de `distinct`, podemos seleccionar columnas.

In [35]:
df6 = spark.createDataFrame([(1,'azul',100),(2,'rojo',110),
                             (1,'azul',150),(4,'verde',202)],
                            ['id','color','tono'])
df6.show()

+---+-----+----+
| id|color|tono|
+---+-----+----+
|  1| azul| 100|
|  2| rojo| 110|
|  1| azul| 150|
|  4|verde| 202|
+---+-----+----+



In [36]:
df6.dropDuplicates(['id','color']).show()

+---+-----+----+
| id|color|tono|
+---+-----+----+
|  1| azul| 100|
|  2| rojo| 110|
|  4|verde| 202|
+---+-----+----+



##### **Función `sort()`**

In [37]:
df4.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('video_id')
).sort('likes').show(5)

+-----+--------+--------------------+
|likes|dislikes|            video_id|
+-----+--------+--------------------+
| NULL|    NULL|ABOUT ARCHITECTUR...|
| NULL|    NULL|                  \n|
| NULL|    NULL|\nTwitter: twitte...|
| NULL|    NULL|  \nDP Joe Victorine|
| NULL|    NULL|                  \n|
+-----+--------+--------------------+
only showing top 5 rows



Ya que nos muestra todo `null` necesitamos ordenar de forma descendente (`F.desc()`) para ver los valores más grandes, ya que resulta más útil.

In [38]:
df4.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('video_id')
).sort(F.desc('likes')).show(5)

+-------+--------+-----------+
|  likes|dislikes|   video_id|
+-------+--------+-----------+
|5613827|  206892|7C2z4GqqS5E|
|5595203|  205565|7C2z4GqqS5E|
|5530568|  200995|7C2z4GqqS5E|
|5486349|  197638|7C2z4GqqS5E|
|5439015|  194172|7C2z4GqqS5E|
+-------+--------+-----------+
only showing top 5 rows



##### **Función `orderBy()`**

In [39]:
df4.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('video_id')
).orderBy(F.col('likes').desc()).show(5)

+-------+--------+-----------+
|  likes|dislikes|   video_id|
+-------+--------+-----------+
|5613827|  206892|7C2z4GqqS5E|
|5595203|  205565|7C2z4GqqS5E|
|5530568|  200995|7C2z4GqqS5E|
|5486349|  197638|7C2z4GqqS5E|
|5439015|  194172|7C2z4GqqS5E|
+-------+--------+-----------+
only showing top 5 rows



Al usar la función `col()` podemos ordenar por más de una columna al mismo tiempo.

In [40]:
df4.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('video_id')
).orderBy(F.col('likes').desc(), F.col('dislikes')).show(5)

+-------+--------+-----------+
|  likes|dislikes|   video_id|
+-------+--------+-----------+
|5613827|  206892|7C2z4GqqS5E|
|5595203|  205565|7C2z4GqqS5E|
|5530568|  200995|7C2z4GqqS5E|
|5486349|  197638|7C2z4GqqS5E|
|5439015|  194172|7C2z4GqqS5E|
+-------+--------+-----------+
only showing top 5 rows



##### **Función `limit()`**

Nos devuelve un límite de resultados concretos.

In [41]:
df4.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('video_id')
).orderBy(F.col('likes').desc()).limit(10).show()

+-------+--------+-----------+
|  likes|dislikes|   video_id|
+-------+--------+-----------+
|5613827|  206892|7C2z4GqqS5E|
|5595203|  205565|7C2z4GqqS5E|
|5530568|  200995|7C2z4GqqS5E|
|5486349|  197638|7C2z4GqqS5E|
|5439015|  194172|7C2z4GqqS5E|
|5386959|  190567|7C2z4GqqS5E|
|5321402|  186331|7C2z4GqqS5E|
|5232318|  179451|7C2z4GqqS5E|
|5150831|  172570|7C2z4GqqS5E|
|5053329|  165853|7C2z4GqqS5E|
+-------+--------+-----------+



##### **Función `withColumn()`**

Como primer parámetro lleva el nombre de la columna. Como segundo parámetro, irá la expresión con la que vamos a construir la columna.

In [42]:
df6 = (df4.withColumn('valoracion', F.col('likes')- F.col('dislikes'))
.select(
    F.col('likes'),
    F.col('dislikes'),
    F.col('valoracion')
))
df6.show(5)

+------+--------+----------+
| likes|dislikes|valoracion|
+------+--------+----------+
| 57527|    2966|     54561|
| 97185|    6146|     91039|
|146033|    5339|    140694|
| 10172|     666|      9506|
|132235|    1989|    130246|
+------+--------+----------+
only showing top 5 rows



In [43]:
df6.printSchema()

root
 |-- likes: long (nullable = true)
 |-- dislikes: long (nullable = true)
 |-- valoracion: long (nullable = true)



##### **Función `withColumnRenamed()`**

Sirve específicamente para cambiar el nombre de una o varias columnas del DataFrame. Si el nombre de la columna que queremos renombrar no existe, spark no llevará a cabo ninguna acción.

In [44]:
df6.withColumnRenamed('valoracion', 'rating').show(5)

+------+--------+------+
| likes|dislikes|rating|
+------+--------+------+
| 57527|    2966| 54561|
| 97185|    6146| 91039|
|146033|    5339|140694|
| 10172|     666|  9506|
|132235|    1989|130246|
+------+--------+------+
only showing top 5 rows



##### **Función `drop()`**

Sirve para eliminar una o varias columnas del dataframe.

In [45]:
df7 = df6.drop('valoracion', 'dislikes')
df7.show(5)

+------+
| likes|
+------+
| 57527|
| 97185|
|146033|
| 10172|
|132235|
+------+
only showing top 5 rows



##### **Función `sample()`**

Nos devuelve una muestra con un % determinado del dataframe. Por tanto, el valor se debe establecer entre 0 y 1. Lleva dos parámetros asociados:

- `fraction=<[0,1]>`: porcentaje de muestra.
- `seed=<num>`: número de semilla.

In [46]:
df8 = df4.sample(fraction=0.8, seed=10)

print(f'El 80% de df4: {df4.count()*0.8} filas.')
print(f'El df8 muestreado al 80%: {df8.count()} filas.')

El 80% de df4: 38509.6 filas.
El df8 muestreado al 80%: 38483 filas.


##### **Función `randomSplit()`**

Es una función de uso común en ML y se usa para separar los datos de train de los datos de test, definiendo un porcentaje para cada conjunto (en total suman 1) y asignandole aleatoriamente valores a cada uno.

Se le pasa una lista con pesos de cada conjunto, así como una semilla.

In [47]:
train, test = df4.randomSplit([0.8,0.2], seed=11)

print(f'Num elementos train: {train.count()}')
print(f'Num elementos test: {test.count()}')

Num elementos train: 38512
Num elementos test: 9625


## Ejercicios

Antes de nada, vamos a cargar el dataframe de `dataCSV` con el nombre `df`.

> Asegúrate de cargar los nombres de las columnas y seleccionar la `,` como delimitador.

In [62]:
# Carga el dataframe de estudiantes.parquet con el nombre df_estudiantes
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 1 – Selección de Columnas

Selecciona las columnas `title`, `channel_title` y `views` para crear un subconjunto del DataFrame. Este tipo de transformación se usa para reducir la dimensionalidad de los datos.

**Resultado esperado:**
Se muestran las tres columnas seleccionadas y las cinco primeras filas.

In [63]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 2 – Filtrado Condicional

Filtra los vídeos con más de **1 millón de visualizaciones**. Este tipo de transformación permite centrarse en los vídeos más populares.

**Resultado esperado:**
Solo aparecen vídeos con `views` superiores a 1.000.000.

In [64]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 3 – Creación de Nueva Columna

Crea una nueva columna `ratio_likes` que indique el porcentaje de likes sobre el total de interacciones (`likes + dislikes`).

**Resultado esperado:**
Nueva columna `ratio_likes` con valores porcentuales como 95.34, 87.12, etc.

In [65]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 4 – Modificación de Columna

Aumenta en un 10% el número de visualizaciones (`views`) para simular un crecimiento estimado.

**Resultado esperado:**
Nueva columna `views_proyectadas` con un valor un 10% mayor que `views`.

In [66]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 5 – Renombrar Columna

Renombra la columna `comment_count` a `n_comentarios` para mejorar la legibilidad del DataFrame.

**Resultado esperado:**
La columna aparece con el nuevo nombre `n_comentarios`.

In [67]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 6 – Filtrado Combinado con Condiciones

Filtra los vídeos que cumplan simultáneamente estas condiciones:
- Más de 1 millón de visualizaciones.
- Más de 50.000 “likes”.
- Que pertenezcan a una categoría diferente de 24.

**Resultado esperado:**
Se muestran vídeos muy vistos y con muchos likes, excluyendo la categoría 24.

In [68]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 7 – Muestreo Aleatorio
Extrae una muestra aleatoria del **10% de los vídeos** para realizar pruebas rápidas o explorar el dataset. Puedes usar la opción `seed` para obtener resultados reproducibles.

**Resultado esperado:**
Devuelve aproximadamente el 10% del total de filas (por ejemplo, si hay 40.000 vídeos, devuelve unas 4.000 filas).

In [69]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 8 – Eliminación de Duplicados

Elimina los vídeos duplicados basados en `video_id`. Este proceso evita redundancia en el análisis.

**Resultado esperado:**
El número total de registros disminuye si existían duplicados.

In [70]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 9 – Ordenación de Datos

Ordena los vídeos de mayor a menor número de likes y muestra los 10 primeros.

**Resultado esperado:**
Lista con los 10 vídeos con más “likes”.

In [71]:
# RELLENA CÓDIGO AQUÍ ---

#### Ejercicio 10 – Transformación Compuesta

Combina varias transformaciones de forma encadenada:

1. Selecciona los vídeos con más de **500.000 visualizaciones**.
2. Calcula el **ratio de likes (likes / views)**.
3. Renombra columnas para que sean más descriptivas
4. Ordena los resultados por **ratio de likes descendente**.
5. Muestra los **5 vídeos con mejor relación de likes por visualización**.

**Resultado esperado:**
Se muestran los 5 vídeos con mejor relación entre likes y visualizaciones, ordenados de mayor a menor ratio.

In [72]:
# RELLENA CÓDIGO AQUÍ ---